# The Corrected Code FOR BVH->Excel DATA WINDOW MARKING 

# 3

In [2]:
"""
Self-windowing of real human RightHand trajectory.

Purpose
-------
Automatically divide a real human-motion trajectory into contiguous,
variable-duration windows in which the acceleration is approximately
constant.

Input
-----
Excel file containing at least:

    Time (s)
    X Position (m)
    Y Position (m)
    Z Position (m)

If velocity/acceleration/jerk columns already exist, they are NOT trusted
for segmentation. They are recomputed internally from the position data.

Output
------
Same original columns +:

    supposed window number

An additional "Window Summary" sheet is also created for inspection.

Important
---------
The script uses the ACTUAL timestamps from the dataset. It does not assume
100 Hz exactly.
"""

from pathlib import Path
import math

import numpy as np
import pandas as pd

from scipy.signal import savgol_filter


# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FILE = Path(
    r"C:\\Users\\samiu\\Desktop\\Kinematics\\Test02_bvh_Global_RArm\\bvh_Meter_Global_RightArm.xlsx"
)

OUTPUT_FILE = INPUT_FILE.parent / "Window Creation" / (
    INPUT_FILE.stem + "_windowed.xlsx"
)


# ------------------------------------------------------------
# Allowed window sizes
# ------------------------------------------------------------

MIN_WINDOW_SEC = 0.30   
MAX_WINDOW_SEC = 0.50   


# ------------------------------------------------------------
# Candidate boundary spacing
#
# 2 samples at ~119 Hz = ~16.8 ms.
#
# Smaller -> finer boundaries but more computation.
# ------------------------------------------------------------

BOUNDARY_STEP_SAMPLES = 2


# ------------------------------------------------------------
# Position smoothing
#
# This smoothing is ONLY for calculating the derivatives used
# by the segmentation algorithm.
#
# Original input values remain untouched.
# ------------------------------------------------------------

SAVGOL_WINDOW_SEC = 0.15
SAVGOL_POLYORDER = 3


# ------------------------------------------------------------
# Cost weights
# ------------------------------------------------------------

W_ACC_VARIATION = 1.00
W_ACC_TREND = 0.35
W_JERK = 0.15
W_VEL_LINEARITY = 0.20


# ------------------------------------------------------------
# Window creation penalty
#
# Larger -> fewer / larger windows
# Smaller -> more / smaller windows
# ------------------------------------------------------------

WINDOW_PENALTY = 0.75


# ------------------------------------------------------------
# Mild penalty for excessively long windows
# ------------------------------------------------------------

LONG_WINDOW_PENALTY = 0.08


# ============================================================
# HELPERS
# ============================================================

def require_columns(df, columns):

    missing = [
        c for c in columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            "Missing required columns:\n"
            + "\n".join(
                f"    {c}"
                for c in missing
            )
        )


def robust_scale(values):

    """
    Robust scale based on MAD.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    median = np.median(values)

    mad = np.median(
        np.abs(values - median)
    )

    scale = 1.4826 * mad

    if scale < 1e-10:

        scale = np.std(values)

    return max(
        float(scale),
        1e-5
    )


def smooth_signal(
    signal,
    window_samples,
    polyorder
):

    """
    Savitzky-Golay smoothing.
    """

    n = len(signal)

    window_samples = max(
        5,
        int(window_samples)
    )

    if window_samples % 2 == 0:
        window_samples += 1

    max_allowed = (
        n
        if n % 2 == 1
        else n - 1
    )

    window_samples = min(
        window_samples,
        max_allowed
    )

    if window_samples <= polyorder:
        return signal.copy()

    output = np.empty_like(
        signal,
        dtype=float
    )

    for k in range(signal.shape[1]):

        output[:, k] = savgol_filter(
            signal[:, k],
            window_length=window_samples,
            polyorder=polyorder,
            mode="interp"
        )

    return output


def calculate_derivatives(
    position,
    dt,
    window_samples,
    polyorder
):

    """
    Calculate velocity, acceleration and jerk directly
    from smoothed position using Savitzky-Golay derivatives.

    This is preferable to differentiating noisy values repeatedly
    with simple finite differences.
    """

    n = len(position)

    window_samples = max(
        7,
        int(window_samples)
    )

    if window_samples % 2 == 0:
        window_samples += 1

    max_allowed = (
        n
        if n % 2 == 1
        else n - 1
    )

    window_samples = min(
        window_samples,
        max_allowed
    )

    if window_samples <= polyorder:
        raise ValueError(
            "Savitzky-Golay window is too small."
        )

    velocity = np.empty_like(
        position,
        dtype=float
    )

    acceleration = np.empty_like(
        position,
        dtype=float
    )

    jerk = np.empty_like(
        position,
        dtype=float
    )

    for k in range(3):

        p = position[:, k]

        velocity[:, k] = savgol_filter(
            p,
            window_length=window_samples,
            polyorder=polyorder,
            deriv=1,
            delta=dt,
            mode="interp"
        )

        acceleration[:, k] = savgol_filter(
            p,
            window_length=window_samples,
            polyorder=polyorder,
            deriv=2,
            delta=dt,
            mode="interp"
        )

        jerk[:, k] = savgol_filter(
            p,
            window_length=window_samples,
            polyorder=polyorder,
            deriv=3,
            delta=dt,
            mode="interp"
        )

    return velocity, acceleration, jerk


# ============================================================
# SEGMENT COST
# ============================================================

def segment_cost(
    acceleration,
    velocity,
    jerk,
    time,
    start,
    end,
    acc_scale,
    jerk_scale,
    velocity_scale
):

    """
    Evaluate how well [start:end) behaves like
    approximately constant acceleration.

    Lower cost = better constant-acceleration window.
    """

    if end - start < 3:
        return np.inf

    A = acceleration[start:end]
    V = velocity[start:end]
    J = jerk[start:end]
    T = time[start:end]

    # --------------------------------------------------------
    # 1. Acceleration variation around its mean
    # --------------------------------------------------------

    A_mean = np.mean(
        A,
        axis=0
    )

    A_residual = A - A_mean

    acc_rmse = np.sqrt(
        np.mean(A_residual ** 2)
    )

    acc_score = (
        acc_rmse
        / acc_scale
    )

    # --------------------------------------------------------
    # 2. Acceleration trend
    #
    # For constant acceleration the slope of A(t) should be
    # approximately zero.
    # --------------------------------------------------------

    local_t = T - T[0]

    t_centered = (
        local_t
        - np.mean(local_t)
    )

    denominator = np.sum(
        t_centered ** 2
    )

    if denominator > 1e-12:

        A_centered = (
            A
            - np.mean(A, axis=0)
        )

        slopes = np.sum(
            t_centered[:, None]
            * A_centered,
            axis=0
        ) / denominator

        acc_trend = np.sqrt(
            np.mean(slopes ** 2)
        )

        acc_trend_score = (
            acc_trend
            / acc_scale
        )

    else:

        acc_trend_score = 0.0

    # --------------------------------------------------------
    # 3. Jerk activity
    #
    # Approximately constant acceleration means relatively
    # low jerk.
    # --------------------------------------------------------

    jerk_rms = math.sqrt(
        np.mean(J ** 2)
    )

    jerk_score = (
        jerk_rms
        / jerk_scale
    )

    # --------------------------------------------------------
    # 4. Velocity linearity
    #
    # Constant acceleration -> approximately linear velocity.
    # --------------------------------------------------------

    V_mean = np.mean(
        V,
        axis=0
    )

    V_centered = (
        V
        - V_mean
    )

    if denominator > 1e-12:

        slopes_v = np.sum(
            t_centered[:, None]
            * V_centered,
            axis=0
        ) / denominator

        V_fit = (
            V_mean
            + t_centered[:, None]
            * slopes_v
        )

        V_residual = (
            V
            - V_fit
        )

        velocity_rmse = np.sqrt(
            np.mean(V_residual ** 2)
        )

        velocity_score = (
            velocity_rmse
            / velocity_scale
        )

    else:

        velocity_score = 0.0

    # --------------------------------------------------------
    # Combined score
    # --------------------------------------------------------

    total = (

        W_ACC_VARIATION
        * acc_score

        +

        W_ACC_TREND
        * acc_trend_score

        +

        W_JERK
        * jerk_score

        +

        W_VEL_LINEARITY
        * velocity_score

    )

    return float(total)


# ============================================================
# LOAD DATASET
# ============================================================

print()
print("=" * 75)
print("REAL HUMAN RIGHTHAND SELF-WINDOWING")
print("=" * 75)

print(
    f"Input : {INPUT_FILE}"
)

print(
    f"Output: {OUTPUT_FILE}"
)

print()


if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"Input file does not exist:\n{INPUT_FILE}"
    )


df = pd.read_excel(
    INPUT_FILE
)


required_columns = [

    "Time (s)",
    "X Position (m)",
    "Y Position (m)",
    "Z Position (m)"

]

require_columns(
    df,
    required_columns
)


if len(df) < 20:

    raise ValueError(
        "Dataset is too short."
    )


# ============================================================
# TIME VALIDATION
# ============================================================

time = pd.to_numeric(
    df["Time (s)"],
    errors="coerce"
).to_numpy(
    dtype=float
)


if np.any(~np.isfinite(time)):

    raise ValueError(
        "Time column contains invalid values."
    )


dt_values = np.diff(time)


if np.any(dt_values <= 0):

    raise ValueError(
        "Time values must be strictly increasing."
    )


median_dt = float(
    np.median(dt_values)
)

sampling_rate = (
    1.0
    / median_dt
)


dt_variation = (
    np.max(dt_values)
    - np.min(dt_values)
)


print(
    f"Rows              : {len(df)}"
)

print(
    f"Start time        : {time[0]:.6f} s"
)

print(
    f"End time          : {time[-1]:.6f} s"
)

print(
    f"Median dt         : {median_dt:.9f} s"
)

print(
    f"Estimated rate    : {sampling_rate:.4f} Hz"
)

print(
    f"dt range          : "
    f"{np.min(dt_values):.9f} - "
    f"{np.max(dt_values):.9f}"
)

print()


# ============================================================
# POSITION DATA
# ============================================================

position = df[
    [
        "X Position (m)",
        "Y Position (m)",
        "Z Position (m)"
    ]
].apply(
    pd.to_numeric,
    errors="coerce"
).to_numpy(
    dtype=float
)


if np.any(~np.isfinite(position)):

    raise ValueError(
        "Position data contain NaN or non-numeric values."
    )


# ============================================================
# SMOOTHING / DERIVATIVES
# ============================================================

smooth_samples = max(
    7,
    int(
        round(
            SAVGOL_WINDOW_SEC
            / median_dt
        )
    )
)

if smooth_samples % 2 == 0:
    smooth_samples += 1


print(
    "Derivative estimation:"
)

print(
    f"  SG window : "
    f"{smooth_samples} samples"
)

print(
    f"  Approx duration : "
    f"{smooth_samples * median_dt:.4f} s"
)

print()


# Calculate velocity / acceleration / jerk
velocity, acceleration, jerk = calculate_derivatives(
    position,
    median_dt,
    smooth_samples,
    SAVGOL_POLYORDER
)


# Additional smoothing for segmentation only.
# Original dataset is never modified.
acceleration_seg = smooth_signal(
    acceleration,
    smooth_samples,
    SAVGOL_POLYORDER
)

jerk_seg = smooth_signal(
    jerk,
    smooth_samples,
    SAVGOL_POLYORDER
)

velocity_seg = smooth_signal(
    velocity,
    smooth_samples,
    SAVGOL_POLYORDER
)


# ============================================================
# GLOBAL ROBUST SCALES
# ============================================================

acc_scale = max(
    robust_scale(acceleration_seg),
    1e-5
)

jerk_scale = max(
    robust_scale(jerk_seg),
    1e-5
)

velocity_scale = max(
    robust_scale(velocity_seg),
    1e-5
)


print("Signal scales:")
print(
    f"  Acceleration scale : {acc_scale:.6f}"
)

print(
    f"  Jerk scale         : {jerk_scale:.6f}"
)

print(
    f"  Velocity scale     : {velocity_scale:.6f}"
)

print()


# ============================================================
# WINDOW SIZE CONVERSION
# ============================================================

n = len(df)

min_samples = max(
    3,
    int(
        math.ceil(
            MIN_WINDOW_SEC
            / median_dt
        )
    )
)

max_samples = max(
    min_samples,
    int(
        math.floor(
            MAX_WINDOW_SEC
            / median_dt
        )
    )
)


print("Window constraints:")
print(
    f"  Minimum : "
    f"{MIN_WINDOW_SEC:.2f} s "
    f"({min_samples} samples)"
)

print(
    f"  Maximum : "
    f"{MAX_WINDOW_SEC:.2f} s "
    f"({max_samples} samples)"
)

print()


# ============================================================
# CANDIDATE BOUNDARIES
# ============================================================

step = max(
    1,
    int(BOUNDARY_STEP_SAMPLES)
)

candidate_indices = list(
    range(
        0,
        n,
        step
    )
)

if candidate_indices[-1] != n:

    candidate_indices.append(n)


candidate_set = set(
    candidate_indices
)


# ============================================================
# DYNAMIC PROGRAMMING
# ============================================================

print(
    "Running dynamic-programming segmentation..."
)

dp = np.full(
    n + 1,
    np.inf
)

previous = np.full(
    n + 1,
    -1,
    dtype=int
)

dp[0] = 0.0


cost_cache = {}


def get_cost(start, end):

    key = (
        start,
        end
    )

    if key not in cost_cache:

        cost_cache[key] = segment_cost(
            acceleration_seg,
            velocity_seg,
            jerk_seg,
            time,
            start,
            end,
            acc_scale,
            jerk_scale,
            velocity_scale
        )

    return cost_cache[key]


# ------------------------------------------------------------
# Candidate endpoints
# ------------------------------------------------------------

endpoints = sorted(
    candidate_set
)

if endpoints[-1] != n:

    endpoints.append(n)


for end in endpoints:

    if end == 0:
        continue

    earliest_start = max(
        0,
        end - max_samples
    )

    latest_start = (
        end - min_samples
    )

    if latest_start < 0:
        continue

    starts = [
        s
        for s in candidate_indices
        if earliest_start <= s <= latest_start
    ]

    # Always allow exact earliest/latest boundaries.
    starts.extend(
        [
            earliest_start,
            latest_start
        ]
    )

    for start in set(starts):

        if start < 0:
            continue

        if end - start < min_samples:
            continue

        if end - start > max_samples:
            continue

        if not np.isfinite(dp[start]):
            continue

        local_cost = get_cost(
            start,
            end
        )

        if not np.isfinite(local_cost):
            continue

        # ----------------------------------------------------
        # Proper window duration
        #
        # For an internal boundary, t[end] is the beginning
        # of the next window.
        #
        # For the final window, use one dt beyond the final
        # observation as the mathematical endpoint.
        # ----------------------------------------------------

        if end < n:

            duration = (
                time[end]
                - time[start]
            )

        else:

            duration = (
                time[-1]
                + median_dt
                - time[start]
            )

        long_penalty = (

            LONG_WINDOW_PENALTY
            * (
                duration
                / MAX_WINDOW_SEC
            ) ** 2

        )

        total_cost = (

            dp[start]
            + local_cost
            + WINDOW_PENALTY
            + long_penalty

        )

        if total_cost < dp[end]:

            dp[end] = total_cost

            previous[end] = start


if not np.isfinite(dp[n]):

    raise RuntimeError(
        "Could not construct a complete segmentation.\n"
        "Try increasing MAX_WINDOW_SEC or decreasing "
        "MIN_WINDOW_SEC."
    )


# ============================================================
# BACKTRACK
# ============================================================

segments = []

current = n


while current > 0:

    start = previous[current]

    if start < 0:

        raise RuntimeError(
            "Backtracking failed."
        )

    segments.append(
        (
            start,
            current
        )
    )

    current = start


segments.reverse()


# ============================================================
# ASSIGN WINDOW NUMBERS
# ============================================================

window_numbers = np.zeros(
    n,
    dtype=int
)


for window_number, (start, end) in enumerate(
    segments,
    start=1
):

    window_numbers[
        start:end
    ] = window_number


if np.any(
    window_numbers == 0
):

    raise RuntimeError(
        "Some samples were not assigned to a window."
    )


# ============================================================
# CREATE OUTPUT
# ============================================================

output_df = df.copy()

output_df[
    "Window Number"
] = window_numbers


# ============================================================
# WINDOW SUMMARY
# ============================================================

summary_rows = []


for window_number, (start, end) in enumerate(
    segments,
    start=1
):

    if end < n:

        end_time = time[end]

    else:

        end_time = (
            time[-1]
            + median_dt
        )

    start_time = time[start]

    duration = (
        end_time
        - start_time
    )

    local_acc = acceleration_seg[
        start:end
    ]

    mean_acc = np.mean(
        local_acc,
        axis=0
    )

    acc_variation = np.sqrt(
        np.mean(
            (
                local_acc
                - mean_acc
            ) ** 2
        )
    )

    local_cost = get_cost(
        start,
        end
    )

    summary_rows.append({

        "Window Number":
            window_number,

        "Start Time (s)":
            start_time,

        "End Time (s)":
            end_time,

        "Window Length (s)":
            duration,

        "Start Frame":
            start,

        "End Frame":
            end - 1,

        "Samples":
            end - start,

        "Mean Acc X (m/s^2)":
            mean_acc[0],

        "Mean Acc Y (m/s^2)":
            mean_acc[1],

        "Mean Acc Z (m/s^2)":
            mean_acc[2],

        "Acceleration Variation":
            acc_variation,

        "Window Cost":
            local_cost
    })


summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# SAVE EXCEL
# ============================================================

print()
print(
    "Writing Excel file..."
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    output_df.to_excel(
        writer,
        sheet_name="Windowed Data",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Window Summary",
        index=False
    )


# ============================================================
# REPORT
# ============================================================

print()
print("=" * 75)
print("WINDOWING COMPLETE")
print("=" * 75)

print(
    f"Output file      : {OUTPUT_FILE}"
)

print(
    f"Number of windows: {len(segments)}"
)

print()


print(
    f"{'Window':>8} "
    f"{'Start':>10} "
    f"{'End':>10} "
    f"{'Length':>10} "
    f"{'Samples':>9} "
    f"{'Cost':>10}"
)

print(
    "-" * 65
)


for row in summary_rows:

    print(
        f"{row['Window Number']:8d} "
        f"{row['Start Time (s)']:10.3f} "
        f"{row['End Time (s)']:10.3f} "
        f"{row['Window Length (s)']:10.3f} "
        f"{row['Samples']:9d} "
        f"{row['Window Cost']:10.4f}"
    )


print()
print("Important:")
print(
    "The original position/velocity/acceleration/jerk columns "
    "are preserved unchanged."
)

print(
    "Derivative signals used for segmentation are calculated "
    "internally from the position trajectory."
)

print(
    "The segmentation uses the actual timestamps in the dataset."
)

print(
    "A window is an approximately constant-acceleration "
    "segment, not an assertion of physically perfect constant acceleration."
)

print("=" * 75)
# 42mins 5.5sec
#2 mins


REAL HUMAN RIGHTHAND SELF-WINDOWING
Input : C:\Users\samiu\Desktop\Kinematics\Test02_bvh_Global_RArm\bvh_Meter_Global_RightArm.xlsx
Output: C:\Users\samiu\Desktop\Kinematics\Test02_bvh_Global_RArm\Window Creation\bvh_Meter_Global_RightArm_windowed.xlsx

Rows              : 62506
Start time        : 0.000000 s
End time          : 525.111599 s
Median dt         : 0.008401113 s
Estimated rate    : 119.0318 Hz
dt range          : 0.008401113 - 0.008401113

Derivative estimation:
  SG window : 19 samples
  Approx duration : 0.1596 s

Signal scales:
  Acceleration scale : 0.195986
  Jerk scale         : 3.657554
  Velocity scale     : 0.037005

Window constraints:
  Minimum : 0.30 s (36 samples)
  Maximum : 0.50 s (59 samples)

Running dynamic-programming segmentation...

Writing Excel file...

WINDOWING COMPLETE
Output file      : C:\Users\samiu\Desktop\Kinematics\Test02_bvh_Global_RArm\Window Creation\bvh_Meter_Global_RightArm_windowed.xlsx
Number of windows: 1153

  Window      Start    